# Lecture 9: Semantic Search & Retrievers

**Course:** NLP with LangChain | **Platform:** Hope to Skill  
**Duration:** ~20 minutes | **Level:** Intermediate  

---

## The Big Picture

In Lecture 7 you learned how to convert text into **embeddings** (numbers).  
In Lecture 8 you stored them in a **vector database** (Qdrant).  
Now it's time to **search** — finding the right documents for any question.

Think of it like a **librarian**:

| Librarian Type | How They Find Books | NLP Equivalent |
|----------------|--------------------|-----------------|
| **Keyword librarian** | Looks for exact title matches | Keyword search |
| **Smart librarian** | Understands what you MEAN and finds related books | **Semantic search** |

> You ask: "books about cars"  
> Keyword librarian: only finds books with "cars" in the title  
> Smart librarian: also finds books about "automobiles", "vehicles", "sedans"

### What You Will Learn

| # | Topic | What You'll Build |
|---|-------|-------------------|
| 1 | Semantic vs keyword search | See the difference yourself |
| 2 | LangChain retrievers | One-line search on any vector store |
| 3 | The `k` parameter | How many results to retrieve |
| 4 | Score threshold | Filter out low-quality matches |
| 5 | MMR (Maximal Marginal Relevance) | Diverse results, no duplicates |
| 6 | Hands-on: compare strategies | Find the best approach for your data |

> **After this lecture:** You'll know how to configure retrieval  
> to get the best, most relevant, and most diverse results  
> for your RAG system.

---

## 0. Environment Setup

Run this cell **once** to install the packages we need.  
No API key required — everything runs locally!

In [2]:
# Install required packages (run once, then you can skip this cell)
%pip install langchain langchain-community langchain-huggingface langchain-qdrant qdrant-client sentence-transformers

Note: you may need to restart the kernel to use updated packages.


---

## 1. Semantic vs Keyword Search

Before we dive into retrievers, let's see **why semantic search matters**.

### Keyword Search = Exact Match Only

```
Query: "automobile maintenance"

Document 1: "Car repair and vehicle servicing guide"  --> NO MATCH (no word "automobile")
Document 2: "Automobile maintenance tips"             --> MATCH (exact word found!)
Document 3: "How to fix your sedan"                   --> NO MATCH
```

### Semantic Search = Meaning Match

```
Query: "automobile maintenance"

Document 1: "Car repair and vehicle servicing guide"  --> MATCH (similar meaning!)
Document 2: "Automobile maintenance tips"             --> MATCH (similar meaning!)
Document 3: "How to fix your sedan"                   --> MATCH (similar meaning!)
```

| Feature | Keyword Search | Semantic Search |
|---------|---------------|----------------|
| Matches | Exact words only | Meaning and concepts |
| "car" finds "automobile"? | No | **Yes** |
| "happy" finds "joyful"? | No | **Yes** |
| Speed | Very fast | Fast (with vector DB) |
| Best for | Product codes, IDs | Natural language questions |

In [ ]:
# ============================================================
# DEMO: Semantic Search in Action
# ============================================================
# Let's build a small knowledge base and show how semantic search
# finds related documents even when the words don't match.

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore

# ============================================================
# QDRANT CLOUD SETUP
# ============================================================
# Use the same credentials from Lecture 8
# If you haven't set up Qdrant Cloud yet:
# 1. Sign up at: https://cloud.qdrant.io/
# 2. Create a cluster (free tier available)
# 3. Copy your cluster URL and API key
# 4. Replace the placeholders below
# ============================================================

QDRANT_URL = ""
QDRANT_API_KEY = ""

# Create sample documents about different topics
# Each Document has page_content (the text) and metadata (extra info)
documents = [
    Document(page_content="Car repair and vehicle servicing guide. Learn how to maintain your automobile.", metadata={"topic": "vehicles"}),
    Document(page_content="Python is a popular programming language for data science and AI applications.", metadata={"topic": "programming"}),
    Document(page_content="The history of sedan cars and how automotive engineering evolved over time.", metadata={"topic": "vehicles"}),
    Document(page_content="Machine learning models learn patterns from large datasets automatically.", metadata={"topic": "AI"}),
    Document(page_content="How to cook Italian pasta from scratch with fresh ingredients.", metadata={"topic": "cooking"}),
    Document(page_content="Truck maintenance and diesel engine repair for commercial vehicles.", metadata={"topic": "vehicles"}),
    Document(page_content="Deep learning uses neural networks with many layers for complex tasks.", metadata={"topic": "AI"}),
    Document(page_content="Baking bread at home: sourdough starter and fermentation tips.", metadata={"topic": "cooking"}),
    Document(page_content="Electric vehicle battery technology and charging infrastructure guide.", metadata={"topic": "vehicles"}),
    Document(page_content="Natural language processing enables computers to understand human text.", metadata={"topic": "AI"}),
]

# Create embeddings + vector store
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = QdrantVectorStore.from_documents(
    documents=documents,
    embedding=embeddings,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    collection_name="search_demo",
)

print(f"Knowledge base: {len(documents)} documents stored in Qdrant Cloud")
print(f"Embedding model: all-MiniLM-L6-v2 (384 dimensions)")
print(f"\nReady for semantic search!")

c:\Users\mhari\miniconda3\envs\demo1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 414.54it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Knowledge base: 10 documents stored in Qdrant Cloud
Embedding model: all-MiniLM-L6-v2 (384 dimensions)

Ready for semantic search!


In [ ]:
# ============================================================
# DEMO: Semantic Search in Action
# ============================================================
# EMBEDDING METHOD: OpenAI Embeddings (PAID, requires API key)
#
# This cell uses OpenAI embeddings (requires API key and costs money)
# For FREE local embeddings, use the previous cell instead
#
# Build the same knowledge base with OpenAI embeddings to compare
# ============================================================

from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore

# ============================================================
# OPENAI SETUP
# ============================================================
# Set your OpenAI API key
# Get your key from: https://platform.openai.com/api-keys
# ============================================================

import os
os.environ["OPENAI_API_KEY"] = ""

# ============================================================
# QDRANT CLOUD SETUP
# ============================================================
QDRANT_URL = ""
QDRANT_API_KEY = ""

# Same documents as before
documents = [
    Document(page_content="Car repair and vehicle servicing guide. Learn how to maintain your automobile.", metadata={"topic": "vehicles"}),
    Document(page_content="Python is a popular programming language for data science and AI applications.", metadata={"topic": "programming"}),
    Document(page_content="The history of sedan cars and how automotive engineering evolved over time.", metadata={"topic": "vehicles"}),
    Document(page_content="Machine learning models learn patterns from large datasets automatically.", metadata={"topic": "AI"}),
    Document(page_content="How to cook Italian pasta from scratch with fresh ingredients.", metadata={"topic": "cooking"}),
    Document(page_content="Truck maintenance and diesel engine repair for commercial vehicles.", metadata={"topic": "vehicles"}),
    Document(page_content="Deep learning uses neural networks with many layers for complex tasks.", metadata={"topic": "AI"}),
    Document(page_content="Baking bread at home: sourdough starter and fermentation tips.", metadata={"topic": "cooking"}),
    Document(page_content="Electric vehicle battery technology and charging infrastructure guide.", metadata={"topic": "vehicles"}),
    Document(page_content="Natural language processing enables computers to understand human text.", metadata={"topic": "AI"}),
]

# Create OpenAI embeddings
print("[PROGRESS] Initializing OpenAI embeddings...")
embeddings_openai = OpenAIEmbeddings(model="text-embedding-3-small")

# Create vector store with OpenAI embeddings
# Note the different collection name to avoid conflicts
print("[PROGRESS] Creating vector store with OpenAI embeddings...")
vectorstore_openai = QdrantVectorStore.from_documents(
    documents=documents,
    embedding=embeddings_openai,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    collection_name="search_demo_openai",
)

print(f"[SUCCESS] Knowledge base created with OpenAI embeddings")
print(f"  Documents: {len(documents)}")
print(f"  Embedding model: text-embedding-3-small (1536 dimensions)")
print(f"  Collection name: search_demo_openai")
print(f"  Cost: ~$0.02 per 1M tokens embedded")
print(f"\n[INFO] This is the PAID version for comparison.")
print(f"[INFO] For FREE embeddings, use the SentenceTransformer version above.")

[PROGRESS] Initializing OpenAI embeddings...
[PROGRESS] Creating vector store with OpenAI embeddings...
[SUCCESS] Knowledge base created with OpenAI embeddings
  Documents: 10
  Embedding model: text-embedding-3-small (1536 dimensions)
  Collection name: search_demo_openai
  Cost: ~$0.02 per 1M tokens embedded

[INFO] This is the PAID version for comparison.
[INFO] For FREE embeddings, use the SentenceTransformer version above.


#### What just happened?

We created a small knowledge base with 10 documents about 3 topics:

| Topic | Documents |
|-------|----------|
| Vehicles | 4 documents (car repair, sedan history, truck maintenance, EVs) |
| AI / Programming | 4 documents (Python, ML, deep learning, NLP) |
| Cooking | 2 documents (pasta, bread) |

We stored them in **Qdrant Cloud** — you can visit your dashboard  
to see the "search_demo" collection!

Notice: no document contains the word "automobile" by itself —  
let's see if semantic search can still find vehicle-related docs!

In [5]:
# ============================================================
# SEARCH: "automobile" — will it find "car" and "vehicle"?
# ============================================================

query = "automobile maintenance"
print(f"Query: '{query}'")
print(f"(No document uses the exact word 'automobile maintenance')\n")

# similarity_search() finds documents with similar MEANING
# k=5 means return the top 5 most similar documents
results = vectorstore.similarity_search(query, k=5)

print(f"--- Semantic Search Results (top 5) ---")
# enumerate() gives us the index (i=0,1,2...) and the document
for i, doc in enumerate(results):
    print(f"\n  #{i + 1} [{doc.metadata['topic']}]:")
    print(f"  {doc.page_content}")

print(f"\nSemantic search found vehicle docs even though")
print(f"we searched for 'automobile' — it understands meaning!")

Query: 'automobile maintenance'
(No document uses the exact word 'automobile maintenance')

--- Semantic Search Results (top 5) ---

  #1 [vehicles]:
  Truck maintenance and diesel engine repair for commercial vehicles.

  #2 [vehicles]:
  Car repair and vehicle servicing guide. Learn how to maintain your automobile.

  #3 [vehicles]:
  The history of sedan cars and how automotive engineering evolved over time.

  #4 [vehicles]:
  Electric vehicle battery technology and charging infrastructure guide.

  #5 [AI]:
  Machine learning models learn patterns from large datasets automatically.

Semantic search found vehicle docs even though
we searched for 'automobile' — it understands meaning!


#### What just happened?

We searched for `"automobile maintenance"` and found documents about:
- **Car repair** — "car" ≈ "automobile" in meaning
- **Vehicle servicing** — "vehicle" ≈ "automobile" in meaning
- **Sedan history** — "sedan" is a type of automobile
- **Truck maintenance** — trucks are vehicles too!

**Keyword search would have found ZERO results** because no document  
contains the exact phrase "automobile maintenance".

> **This is the power of semantic search:**  
> It matches meaning, not words. This is what makes RAG work so well.

---

## 2. The Retriever Abstraction in LangChain

LangChain provides a **universal search interface** called a **Retriever**.  
Instead of calling different search methods for different databases,  
you just use one consistent interface.

### The Magic of `.as_retriever()`

```python
# This ONE line works with ANY vector store!
retriever = vectorstore.as_retriever()
```

No matter which vector store you use — Qdrant, Pinecone, Chroma,  
FAISS, Weaviate — the code is the **same**:

| Vector Store | Creating the Retriever | Searching |
|-------------|----------------------|----------|
| Qdrant | `qdrant_store.as_retriever()` | `retriever.invoke(query)` |
| Pinecone | `pinecone_store.as_retriever()` | `retriever.invoke(query)` |
| Chroma | `chroma_store.as_retriever()` | `retriever.invoke(query)` |
| FAISS | `faiss_store.as_retriever()` | `retriever.invoke(query)` |

**Same `.invoke(query)` call every time!**
This means you can switch vector stores without changing your RAG code.

> **Think of it like a universal remote control.**  
> You press the same buttons no matter which TV you have.

In [6]:
# ============================================================
# CREATE: Your First Retriever
# ============================================================

# .as_retriever() converts a vector store into a retriever
# This is the standard way to search in LangChain
retriever = vectorstore.as_retriever()

# .invoke() runs the search — returns a list of Document objects
query = "How does artificial intelligence work?"
results = retriever.invoke(query)

print(f"Query: '{query}'")
print(f"Results: {len(results)} documents retrieved\n")

# Show the results
for i, doc in enumerate(results):
    print(f"  #{i + 1} [{doc.metadata['topic']}]: {doc.page_content[:80]}...")

print(f"\nThat's it! One line to create, one line to search.")
print(f"retriever = vectorstore.as_retriever()")
print(f"results   = retriever.invoke(query)")

Query: 'How does artificial intelligence work?'
Results: 4 documents retrieved

  #1 [AI]: Deep learning uses neural networks with many layers for complex tasks....
  #2 [AI]: Natural language processing enables computers to understand human text....
  #3 [AI]: Machine learning models learn patterns from large datasets automatically....
  #4 [programming]: Python is a popular programming language for data science and AI applications....

That's it! One line to create, one line to search.
retriever = vectorstore.as_retriever()
results   = retriever.invoke(query)


#### What just happened?

We created a retriever in **one line** and searched in **one line**:

```python
retriever = vectorstore.as_retriever()   # Create
results   = retriever.invoke(query)      # Search
```

The retriever returned a list of `Document` objects — the same objects  
from Lecture 5 with `page_content` and `metadata`.

By default, the retriever returns **4 documents** (k=4).  
Let's learn how to control this.

---

## 3. The `k` Parameter: How Many Results?

The **`k` parameter** controls how many documents the retriever returns.  
Choosing the right `k` is important!

### The Trade-Off

```
k too LOW (k=1-2):             k too HIGH (k=15+):
                                
  + Very focused                  + Won't miss anything
  - Might miss crucial info       - Lots of noise/junk
  - One bad chunk = bad answer    - Confuses the LLM
                                  - Wastes context window
                                  - Slower + more expensive

           k=3-5 is the sweet spot for most use cases
```

| k Value | When to Use |
|---------|------------|
| k=2-3 | Simple factual questions ("When was X?") |
| **k=4-5** | **General purpose (start here!)** |
| k=7-10 | Complex questions needing broad context |
| k=10+ | Only with reranking (Lecture 13) |

In [7]:
# ============================================================
# COMPARE: Different k Values
# ============================================================
# Let's search with k=2, k=5, and k=8 to see the difference.

query = "Tell me about artificial intelligence"
print(f"Query: '{query}'\n")

# Test 3 different k values
k_values = [2, 5, 8]

for k in k_values:
    # search_kwargs is a dictionary of search parameters
    # {"k": k} tells the retriever how many documents to return
    retriever_k = vectorstore.as_retriever(search_kwargs={"k": k})
    results = retriever_k.invoke(query)

    print(f"--- k={k} ({len(results)} results) ---")
    for i, doc in enumerate(results):
        # [:60] shows first 60 characters of each document
        print(f"  #{i + 1} [{doc.metadata['topic']}]: {doc.page_content[:60]}...")
    print()

Query: 'Tell me about artificial intelligence'

--- k=2 (2 results) ---
  #1 [AI]: Deep learning uses neural networks with many layers for comp...
  #2 [programming]: Python is a popular programming language for data science an...

--- k=5 (5 results) ---
  #1 [AI]: Deep learning uses neural networks with many layers for comp...
  #2 [programming]: Python is a popular programming language for data science an...
  #3 [AI]: Natural language processing enables computers to understand ...
  #4 [AI]: Machine learning models learn patterns from large datasets a...
  #5 [vehicles]: Truck maintenance and diesel engine repair for commercial ve...

--- k=8 (8 results) ---
  #1 [AI]: Deep learning uses neural networks with many layers for comp...
  #2 [programming]: Python is a popular programming language for data science an...
  #3 [AI]: Natural language processing enables computers to understand ...
  #4 [AI]: Machine learning models learn patterns from large datasets a...
  #5 [vehicles]: Tru

#### What just happened?

We searched for the same question with 3 different `k` values:

| k | What You Got |
|---|-------------|
| **k=2** | Only the 2 most relevant — focused but might miss something |
| **k=5** | Good coverage — the top 5 most relevant documents |
| **k=8** | Lots of results — but the last few may be about cooking or vehicles! |

Notice how **higher k brings in less relevant results**?  
With k=8, you might get cooking or vehicle docs mixed in with AI docs.

> **Start with k=4 or k=5.** Only increase if your answers  
> are missing important information (low RAGAS context_recall).

---

## 4. Score Threshold: Quality Filter

What if you only want results that are **actually relevant**?  
The **score threshold** filters out low-quality matches.

### How It Works

```
Without threshold:              With threshold (0.3):

  Doc A: 0.85 (great)  ✓          Doc A: 0.85 (great)  ✓
  Doc B: 0.72 (good)   ✓          Doc B: 0.72 (good)   ✓
  Doc C: 0.45 (okay)   ✓          Doc C: 0.45 (okay)   ✓
  Doc D: 0.15 (junk)   ✓          Doc D: 0.15 (junk)   ✗ FILTERED OUT
```

| Threshold | Effect |
|-----------|--------|
| **No threshold** | Returns exactly k results, even if some are junk |
| **Low (0.2-0.3)** | Removes only the worst matches |
| **Medium (0.4-0.5)** | Returns only reasonably relevant results |
| **High (0.7+)** | Only returns very similar documents (may return few/none) |

In [8]:
# ============================================================
# DEMO: Score Threshold Filtering
# ============================================================
# First, let's see the actual similarity scores for a query.

query = "How does artificial intelligence work?"
print(f"Query: '{query}'\n")

# similarity_search_with_score() returns documents AND their scores
# This lets us see how similar each document is to the query
results_with_scores = vectorstore.similarity_search_with_score(query, k=8)

print(f"--- All 8 Results with Similarity Scores ---")
for i, (doc, score) in enumerate(results_with_scores):
    # Lower score = more similar (for Qdrant distance metrics)
    # We show the raw score so students can see the range
    # [:55] shows first 55 characters of the document
    print(f"  #{i + 1} (score: {score:.4f}) [{doc.metadata['topic']}]: {doc.page_content[:55]}...")

print(f"\nNotice how the scores get worse (higher) as we go down.")
print(f"The last results are about cooking — clearly not relevant!")

Query: 'How does artificial intelligence work?'

--- All 8 Results with Similarity Scores ---
  #1 (score: 0.4742) [AI]: Deep learning uses neural networks with many layers for...
  #2 (score: 0.4294) [AI]: Natural language processing enables computers to unders...
  #3 (score: 0.3866) [AI]: Machine learning models learn patterns from large datas...
  #4 (score: 0.3167) [programming]: Python is a popular programming language for data scien...
  #5 (score: 0.1152) [vehicles]: Truck maintenance and diesel engine repair for commerci...
  #6 (score: 0.1067) [cooking]: How to cook Italian pasta from scratch with fresh ingre...
  #7 (score: 0.0827) [vehicles]: The history of sedan cars and how automotive engineerin...
  #8 (score: 0.0363) [vehicles]: Electric vehicle battery technology and charging infras...

Notice how the scores get worse (higher) as we go down.
The last results are about cooking — clearly not relevant!


#### What just happened?

We used `similarity_search_with_score()` to see the **actual scores**:

- **Top results** have low distance scores (very similar to the query)
- **Bottom results** have high distance scores (not related at all)

> **Note on scores:** Qdrant returns **distance** (lower = more similar),  
> not similarity (higher = more similar). This is just a convention —  
> the ranking is the same either way.

A score threshold would remove the bottom results automatically,  
so only genuinely relevant documents reach the LLM.

In [9]:
# ============================================================
# DEMO: Using Score Threshold with a Retriever
# ============================================================
# search_type="similarity_score_threshold" enables score filtering

# Create a retriever with a score threshold
# Only documents with similarity >= 0.5 will be returned
threshold_retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.5},
)

# Search — only high-quality matches will be returned
query = "How does artificial intelligence work?"
# query = "what is artificial intelligence?"
results = threshold_retriever.invoke(query)

print(f"Query: '{query}'")
print(f"Threshold: 0.5 (only return if similarity >= 50%)")
print(f"Results returned: {len(results)}\n")

for i, doc in enumerate(results):
    print(f"  #{i + 1} [{doc.metadata['topic']}]: {doc.page_content[:70]}...")

if len(results) < 8:
    print(f"\nOnly {len(results)} results passed the threshold!")
    print(f"The irrelevant cooking/vehicle docs were filtered out.")

Query: 'How does artificial intelligence work?'
Threshold: 0.5 (only return if similarity >= 50%)
Results returned: 4

  #1 [AI]: Deep learning uses neural networks with many layers for complex tasks....
  #2 [AI]: Natural language processing enables computers to understand human text...
  #3 [AI]: Machine learning models learn patterns from large datasets automatical...
  #4 [programming]: Python is a popular programming language for data science and AI appli...

Only 4 results passed the threshold!
The irrelevant cooking/vehicle docs were filtered out.


#### What just happened?

We created a retriever with `score_threshold=0.5`:

| Without Threshold | With Threshold (0.5) |
|-------------------|---------------------|
| Returns exactly k docs | Returns only docs above the threshold |
| Includes junk at the bottom | Filters out low-quality matches |
| Always returns something | May return 0 results if nothing is relevant |

### When to Use Score Threshold

| Use Case | Use Threshold? |
|----------|---------------|
| RAG chatbot (always need an answer) | Usually no — use k instead |
| Document search (quality matters) | Yes — filter out noise |
| Large mixed-topic collections | Yes — avoid cross-topic results |

> **Tip:** Start without a threshold and add one only if you see  
> irrelevant results polluting your RAG answers.

---

## 5. The Problem: Redundant Results

There's a sneaky problem with standard similarity search:  
**it can return 5 documents that all say the same thing!**

### Example

```
Query: "What is Python used for?"

Standard similarity search (top 5):
  #1: "Python is great for data science"           (score: 0.92)
  #2: "Python is excellent for data analysis"       (score: 0.91) <-- SAME IDEA!
  #3: "Python is popular in data science"           (score: 0.90) <-- SAME IDEA!
  #4: "Python is widely used for data work"         (score: 0.89) <-- SAME IDEA!
  #5: "Python is the top data science language"     (score: 0.88) <-- SAME IDEA!
```

All 5 results say the same thing! The user learns nothing new  
after the first result. This is a **waste of the context window**.

### What We Actually Want

```
Better results (diverse):
  #1: "Python is great for data science"            (data science)
  #2: "Python is used to build web applications"    (web dev)
  #3: "Python automates repetitive tasks"           (automation)
  #4: "Python is popular for machine learning"      (ML)
  #5: "Python scripts can process files and data"   (scripting)
```

Each result adds **new information**. This gives the LLM a much  
richer context to answer from.

> **This is the problem MMR solves.**

In [10]:
# ============================================================
# DEMO: The Redundancy Problem
# ============================================================
# Let's create a dataset with many similar documents
# to show how standard search returns redundant results.

# Documents with deliberate overlap — many say similar things
redundant_docs = [
    Document(page_content="NLP is a branch of artificial intelligence that processes human language.", metadata={"id": 1}),
    Document(page_content="Natural language processing is an AI field focused on understanding text.", metadata={"id": 2}),
    Document(page_content="NLP uses AI to analyze, understand, and generate human language.", metadata={"id": 3}),
    Document(page_content="The transformer architecture revolutionized NLP with self-attention.", metadata={"id": 4}),
    Document(page_content="BERT is a bidirectional transformer model for understanding text.", metadata={"id": 5}),
    Document(page_content="GPT models generate text by predicting the next word in a sequence.", metadata={"id": 6}),
    Document(page_content="LangChain helps developers build applications with large language models.", metadata={"id": 7}),
    Document(page_content="RAG retrieves relevant documents and uses them to generate answers.", metadata={"id": 8}),
]

# Build a new vector store with these documents
# Using the same Qdrant Cloud credentials
redundant_vectorstore = QdrantVectorStore.from_documents(
    documents=redundant_docs,
    embedding=embeddings,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    collection_name="redundancy_demo",
)

# Standard search — may return redundant results
query = "What is NLP?"
standard_results = redundant_vectorstore.similarity_search(query, k=5)

print(f"Query: '{query}'\n")
print(f"--- Standard Search (top 5) ---")
for i, doc in enumerate(standard_results):
    print(f"  #{i + 1} [doc {doc.metadata['id']}]: {doc.page_content}")

print(f"\nNotice: The first 3 results all say the same thing!")
print(f"They're about 'NLP is AI for language' — just worded differently.")

Query: 'What is NLP?'

--- Standard Search (top 5) ---
  #1 [doc 1]: NLP is a branch of artificial intelligence that processes human language.
  #2 [doc 3]: NLP uses AI to analyze, understand, and generate human language.
  #3 [doc 2]: Natural language processing is an AI field focused on understanding text.
  #4 [doc 5]: BERT is a bidirectional transformer model for understanding text.
  #5 [doc 4]: The transformer architecture revolutionized NLP with self-attention.

Notice: The first 3 results all say the same thing!
They're about 'NLP is AI for language' — just worded differently.


#### What just happened?

Standard similarity search returned the **3 most similar documents first** —  
but they all say essentially the same thing: "NLP is AI for language."

Documents about transformers, BERT, GPT, and LangChain would have  
added much more useful information to the answer.

> **The problem:** Standard search ranks by similarity only.  
> It doesn't care if results are redundant.  
> **The fix:** MMR — Maximal Marginal Relevance.

---

## 6. MMR: Maximal Marginal Relevance

MMR solves the redundancy problem by balancing **two goals**:

1. **Relevance** — the document should be similar to the query
2. **Diversity** — the document should be DIFFERENT from results already selected

### How MMR Works (Simple Version)

```
Step 1: Find the MOST relevant document         → Pick it
Step 2: Find the next document that is:
         - Still relevant to the query           (relevance)
         - BUT different from doc #1             (diversity)
Step 3: Repeat — always balancing relevance AND diversity
```

### The Lambda Parameter

MMR uses a parameter called **lambda** (`lambda_mult` in LangChain)  
to control the balance:

| Lambda Value | Behavior | Best For |
|-------------|----------|----------|
| **1.0** | Pure relevance (same as standard search) | Focused factual queries |
| **0.7** | Mostly relevance, some diversity | General purpose |
| **0.5** | Equal balance (recommended default) | Broad questions |
| **0.3** | Mostly diversity, some relevance | Exploratory research |
| **0.0** | Pure diversity (ignores relevance!) | Never use this |

### MMR Parameters

| Parameter | What It Does | Example |
|-----------|-------------|--------|
| `k` | How many documents to **return** | k=5 → return 5 results |
| `fetch_k` | How many documents to **consider** | fetch_k=20 → look at 20, pick best 5 |
| `lambda_mult` | Balance relevance vs diversity | 0.5 = balanced |

> **Think of it like casting a movie:**  
> `fetch_k=20` is the audition pool (20 actors try out)  
> `k=5` is the final cast (pick the 5 best AND most different actors)  
> `lambda_mult=0.5` controls if you prefer the best actors  
> or the most diverse cast

In [11]:
# ============================================================
# DEMO: Standard Search vs MMR Search
# ============================================================
# Let's compare standard and MMR results side by side.

query = "What is NLP?"
print(f"Query: '{query}'\n")

# --- Standard Search ---
standard_results = redundant_vectorstore.similarity_search(query, k=5)

print("--- Standard Search (relevance only) ---")
for i, doc in enumerate(standard_results):
    print(f"  #{i + 1} [doc {doc.metadata['id']}]: {doc.page_content[:70]}...")

# --- MMR Search ---
# max_marginal_relevance_search() uses the MMR algorithm
# k=5: return 5 results
# fetch_k=8: consider all 8 documents as candidates
# lambda_mult=0.5: balanced between relevance and diversity
mmr_results = redundant_vectorstore.max_marginal_relevance_search(
    query,
    k=5,
    fetch_k=8,
    lambda_mult=0.5,
)

print("\n--- MMR Search (relevance + diversity) ---")
for i, doc in enumerate(mmr_results):
    print(f"  #{i + 1} [doc {doc.metadata['id']}]: {doc.page_content[:70]}...")

# Show the difference
standard_ids = [doc.metadata["id"] for doc in standard_results]
mmr_ids = [doc.metadata["id"] for doc in mmr_results]
print(f"\nStandard doc IDs: {standard_ids}")
print(f"MMR doc IDs:      {mmr_ids}")
print(f"\nMMR brings in MORE diverse documents!")

Query: 'What is NLP?'

--- Standard Search (relevance only) ---
  #1 [doc 1]: NLP is a branch of artificial intelligence that processes human langua...
  #2 [doc 3]: NLP uses AI to analyze, understand, and generate human language....
  #3 [doc 2]: Natural language processing is an AI field focused on understanding te...
  #4 [doc 5]: BERT is a bidirectional transformer model for understanding text....
  #5 [doc 4]: The transformer architecture revolutionized NLP with self-attention....

--- MMR Search (relevance + diversity) ---
  #1 [doc 1]: NLP is a branch of artificial intelligence that processes human langua...
  #2 [doc 8]: RAG retrieves relevant documents and uses them to generate answers....
  #3 [doc 6]: GPT models generate text by predicting the next word in a sequence....
  #4 [doc 5]: BERT is a bidirectional transformer model for understanding text....
  #5 [doc 7]: LangChain helps developers build applications with large language mode...

Standard doc IDs: [1, 3, 2, 5, 4]
M

#### What just happened?

We compared standard and MMR search:

| Standard Search | MMR Search |
|----------------|------------|
| Top 3 results all say "NLP = AI for language" | First result is most relevant |
| Redundant — same info repeated | Each result adds **new** information |
| Wastes context window | Efficient use of context window |

MMR should bring in documents about transformers, BERT, GPT, or  
LangChain — not just 5 versions of "NLP is AI."

> **MMR gives the LLM a richer context to answer from,**  
> which usually leads to better, more complete answers.

In [12]:
# ============================================================
# DEMO: MMR with Different Lambda Values
# ============================================================
# Let's see how lambda_mult affects the balance.

query = "What is NLP?"
print(f"Query: '{query}'\n")

# Test 3 different lambda values
lambda_values = [
    (1.0, "Pure relevance (same as standard)"),
    (0.5, "Balanced (recommended)"),
    (0.2, "Heavy diversity"),
]

for lam, description in lambda_values:
    results = redundant_vectorstore.max_marginal_relevance_search(
        query, k=4, fetch_k=8, lambda_mult=lam,
    )

    # Collect the document IDs to see which docs were picked
    ids = [doc.metadata["id"] for doc in results]

    print(f"Lambda={lam} — {description}")
    print(f"  Doc IDs returned: {ids}")
    for i, doc in enumerate(results):
        # [:60] shows first 60 characters
        print(f"    #{i + 1}: {doc.page_content[:60]}...")
    print()

Query: 'What is NLP?'

Lambda=1.0 — Pure relevance (same as standard)
  Doc IDs returned: [1, 7, 8, 6]
    #1: NLP is a branch of artificial intelligence that processes hu...
    #2: LangChain helps developers build applications with large lan...
    #3: RAG retrieves relevant documents and uses them to generate a...
    #4: GPT models generate text by predicting the next word in a se...

Lambda=0.5 — Balanced (recommended)
  Doc IDs returned: [1, 8, 6, 5]
    #1: NLP is a branch of artificial intelligence that processes hu...
    #2: RAG retrieves relevant documents and uses them to generate a...
    #3: GPT models generate text by predicting the next word in a se...
    #4: BERT is a bidirectional transformer model for understanding ...

Lambda=0.2 — Heavy diversity
  Doc IDs returned: [1, 3, 5, 4]
    #1: NLP is a branch of artificial intelligence that processes hu...
    #2: NLP uses AI to analyze, understand, and generate human langu...
    #3: BERT is a bidirectional transformer 

#### What just happened?

We tested 3 lambda values to see the spectrum:

| Lambda | What It Did |
|--------|------------|
| **1.0** | Same as standard search — all about "NLP is AI" |
| **0.5** | Balanced — mixes NLP basics with transformers/BERT/GPT |
| **0.2** | Maximum diversity — covers the most different topics |

### Which Lambda Should You Use?

| Situation | Lambda | Why |
|-----------|--------|-----|
| Simple factual Q&A | 0.7 - 1.0 | Need the most relevant docs |
| General RAG chatbot | **0.5** | Good balance (start here) |
| Research / exploration | 0.3 - 0.5 | Want broad coverage |

> **Start with lambda=0.5 (balanced).** Adjust up if answers  
> lack focus, adjust down if answers are repetitive.

---

## 7. Using MMR with a Retriever

You can configure MMR directly in the retriever using `search_type="mmr"`.  
This makes it easy to plug MMR into any RAG chain.

In [13]:
# ============================================================
# BUILD: MMR Retriever (production-ready)
# ============================================================
# This is how you'd use MMR in a real RAG system.

# Standard retriever (similarity only)
standard_retriever = redundant_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

# MMR retriever (relevance + diversity)
mmr_retriever = redundant_vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,              # Return 4 documents
        "fetch_k": 8,        # Consider 8 candidates
        "lambda_mult": 0.5,  # Balanced relevance/diversity
    },
)

print("Two retrievers created:")
print("  1. Standard: search_type='similarity', k=4")
print("  2. MMR:      search_type='mmr', k=4, fetch_k=8, lambda=0.5")
print()

# Compare them
query = "What is NLP?"

standard_docs = standard_retriever.invoke(query)
mmr_docs = mmr_retriever.invoke(query)

print(f"Query: '{query}'\n")

print("Standard retriever:")
for i, doc in enumerate(standard_docs):
    print(f"  #{i + 1}: {doc.page_content[:65]}...")

print("\nMMR retriever:")
for i, doc in enumerate(mmr_docs):
    print(f"  #{i + 1}: {doc.page_content[:65]}...")

print(f"\nMMR retriever gives more diverse results!")
print(f"Use this in your RAG chain for better answers.")

Two retrievers created:
  1. Standard: search_type='similarity', k=4
  2. MMR:      search_type='mmr', k=4, fetch_k=8, lambda=0.5

Query: 'What is NLP?'

Standard retriever:
  #1: NLP is a branch of artificial intelligence that processes human l...
  #2: NLP uses AI to analyze, understand, and generate human language....
  #3: Natural language processing is an AI field focused on understandi...
  #4: BERT is a bidirectional transformer model for understanding text....

MMR retriever:
  #1: NLP is a branch of artificial intelligence that processes human l...
  #2: RAG retrieves relevant documents and uses them to generate answer...
  #3: GPT models generate text by predicting the next word in a sequenc...
  #4: BERT is a bidirectional transformer model for understanding text....

MMR retriever gives more diverse results!
Use this in your RAG chain for better answers.


#### What just happened?

We created two retrievers and compared them:

```python
# Standard — just similarity
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

# MMR — similarity + diversity
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 8, "lambda_mult": 0.5},
)
```

Both use the same `.invoke(query)` call. The only difference is the  
`search_type` parameter. You can swap between them without changing  
any other part of your RAG chain!

> **In Lecture 10, you'll plug this retriever directly into**  
> **a full RAG chain using LCEL.** The retriever is a drop-in component.

---

## 8. When to Use What?

Here's your decision guide:

| Strategy | When to Use | Code |
|----------|------------|------|
| **Standard (similarity)** | Small focused collections, factual Q&A | `search_type="similarity"` |
| **Score threshold** | Large mixed-topic collections, quality matters | `search_type="similarity_score_threshold"` |
| **MMR** | Large collections, many similar docs, broad queries | `search_type="mmr"` |

### Quick Decision Flowchart

```
Are your results redundant (same info repeated)?
  └─ YES → Use MMR (lambda_mult=0.5)
  └─ NO  → Standard search is fine

Are irrelevant documents showing up?
  └─ YES → Add score_threshold
  └─ NO  → Don't need it

Are answers missing important info?
  └─ YES → Increase k (try k=7-10)
  └─ NO  → k=4-5 is good
```

---

## 9. Hands-On: Compare All Strategies

Let's use our original knowledge base (10 documents) and test  
all three retrieval strategies on the same query.

In [14]:
# ============================================================
# COMPARE: Standard vs Threshold vs MMR
# ============================================================
# Using our original 10-document knowledge base.

query = "Tell me about machine learning and AI"
print(f"Query: '{query}'")
print(f"Knowledge base: {len(documents)} documents\n")

# Strategy 1: Standard similarity (k=5)
ret_standard = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

# Strategy 2: Score threshold
ret_threshold = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.5},
)

# Strategy 3: MMR (balanced)
ret_mmr = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 10, "lambda_mult": 0.5},
)

# Run all 3 strategies
strategies = [
    ("Standard (k=5)", ret_standard),
    ("Threshold (0.5)", ret_threshold),
    ("MMR (lambda=0.5)", ret_mmr),
]

for name, ret in strategies:
    results = ret.invoke(query)
    print(f"--- {name}: {len(results)} results ---")
    for i, doc in enumerate(results):
        # [:65] shows first 65 characters of the document
        print(f"  #{i + 1} [{doc.metadata['topic']}]: {doc.page_content[:65]}...")
    print()

Query: 'Tell me about machine learning and AI'
Knowledge base: 10 documents

--- Standard (k=5): 5 results ---
  #1 [AI]: Machine learning models learn patterns from large datasets automa...
  #2 [AI]: Deep learning uses neural networks with many layers for complex t...
  #3 [programming]: Python is a popular programming language for data science and AI ...
  #4 [AI]: Natural language processing enables computers to understand human...
  #5 [vehicles]: Truck maintenance and diesel engine repair for commercial vehicle...

--- Threshold (0.5): 4 results ---
  #1 [AI]: Machine learning models learn patterns from large datasets automa...
  #2 [AI]: Deep learning uses neural networks with many layers for complex t...
  #3 [programming]: Python is a popular programming language for data science and AI ...
  #4 [AI]: Natural language processing enables computers to understand human...

--- MMR (lambda=0.5): 5 results ---
  #1 [AI]: Machine learning models learn patterns from large datasets au

#### What just happened?

We compared all 3 strategies on the same query:

| Strategy | What It Returned |
|----------|------------------|
| **Standard** | Top 5 most similar docs (always returns exactly 5) |
| **Threshold** | Only docs with similarity >= 0.5 (may be fewer than 5) |
| **MMR** | 5 docs that are relevant AND diverse (fewer duplicates) |

Look at the **topics** in each result set:
- Standard may have all AI docs (relevant but repetitive)
- MMR should include more variety while staying on topic
- Threshold may return fewer results if some didn't meet the cutoff

> **For your RAG system in Lecture 10,** we'll use standard search  
> with k=5 as the default. If you notice redundancy, switch to MMR.

---

## 10. Mini Challenges

### Challenge 1: Find the Best k
Test k values of 2, 4, 6, and 8 with the query  
`"What is natural language processing?"`.  
At what point do you start seeing irrelevant results?  
What's the best k for this knowledge base?

### Challenge 2: The MMR Detective
Use the `redundant_vectorstore` and search for `"Tell me about language models"`.  
Compare `lambda_mult=1.0` (pure relevance) with `lambda_mult=0.3`  
(heavy diversity). What different documents does MMR surface  
that standard search misses?

### Challenge 3: Build a Smart Retriever
Create a retriever that uses MMR with `k=4`, `fetch_k=8`,  
and `lambda_mult=0.5`. Test it with 3 different queries.  
For each query, compare the MMR results with standard results  
and explain which gives better context for answering the question.

> **Hint for Challenge 3:** Use `vectorstore.as_retriever(search_type="mmr", ...)`  
> for MMR, and `vectorstore.as_retriever()` for standard.

In [ ]:
# ============================================================
# Challenge 1: Find the Best k
# ============================================================
# Test k=2, 4, 6, 8 with "What is natural language processing?"
# At what k do irrelevant results start appearing?

# Your code here:


In [ ]:
# ============================================================
# Challenge 2: The MMR Detective
# ============================================================
# Compare lambda=1.0 vs lambda=0.3 on redundant_vectorstore
# Query: "Tell me about language models"

# Your code here:


In [ ]:
# ============================================================
# Challenge 3: Build a Smart Retriever
# ============================================================
# Create MMR retriever (k=4, fetch_k=8, lambda=0.5)
# Test with 3 queries, compare with standard retriever

# Your code here:


---

## 11. Quick Reference: Retriever Cheat Sheet

### Three Retrieval Strategies

| Strategy | Code | Best For |
|----------|------|----------|
| **Standard** | `search_type="similarity"` | Simple factual queries |
| **Threshold** | `search_type="similarity_score_threshold"` | Filtering noise |
| **MMR** | `search_type="mmr"` | Diverse results, large collections |

### Quick Code Patterns

```python
# Standard retriever (most common)
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5},
)

# Score threshold retriever
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.5},
)

# MMR retriever (recommended for large collections)
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 20, "lambda_mult": 0.5},
)

# Use any retriever the same way
results = retriever.invoke("your question here")
```

### Parameter Guide

| Parameter | What It Controls | Default | Range |
|-----------|-----------------|---------|-------|
| `k` | Number of results to return | 4 | 2-10 |
| `score_threshold` | Minimum similarity score | None | 0.0-1.0 |
| `fetch_k` | MMR candidate pool size | 20 | 2× to 5× k |
| `lambda_mult` | MMR relevance vs diversity | 0.5 | 0.0-1.0 |

### Decision Guide

```
Redundant results? → Use MMR
Irrelevant noise?  → Add score_threshold or lower k
Missing info?      → Increase k
Not sure?          → Start with Standard k=5, evaluate, adjust
```

---

## 12. Key Takeaways

1. **Semantic search matches meaning**, not words — "car" finds "automobile"
2. **LangChain retrievers** give you one interface for any vector store — `.as_retriever()` + `.invoke()`
3. **`k` parameter** controls how many results — start with k=4-5
4. **Score threshold** filters out low-quality matches when you need quality over quantity
5. **MMR prevents redundancy** by balancing relevance and diversity
6. **Start simple, then tune:** standard k=5 → add MMR if redundant → add threshold if noisy

### The Pipeline So Far

```
Load (L5) → Split (L6) → Embed (L7) → Store (L8) → Retrieve (L9) → ???
                                                         ^
                                                    You are here!
```

### Next Lecture

**Lecture 10: Building Vanilla RAG with LCEL** — Now you'll connect  
the retriever to an LLM and build a complete question-answering system!  
The retriever you just learned about becomes one piece of the LCEL chain.

---

*Hope to Skill — Building the future, one skill at a time.*

---

## Appendix: PEP 8 Style Rules Used in This Notebook

All code in this notebook follows Python's PEP 8 style guide.  
Here are the rules applied, with examples from the code above.

### Naming Conventions

| Rule | Convention | Example from This Notebook |
|------|-----------|---------------------------|
| Variables & functions | `snake_case` | `mmr_retriever`, `standard_results`, `lambda_values` |
| Constants | `UPPER_CASE` | None used (configurations are variable) |
| Classes | `PascalCase` | `QdrantVectorStore`, `HuggingFaceEmbeddings`, `Document` (from libraries) |

### Import Rules

| Rule ID | Rule | Example |
|---------|------|---------|
| E401 | One import per line | `from langchain_core.documents import Document` |
| E402 | Imports at the top of each section | All imports at cell top |
| — | Group: stdlib then third-party then local | `langchain_core` then `langchain_qdrant` |

### Whitespace & Formatting

| Rule ID | Rule | Example |
|---------|------|---------|
| E225 | Spaces around operators | `i + 1`, `score >= 0.5` |
| E231 | Space after commas | `search_kwargs={"k": 5, "fetch_k": 20}` |
| E265 | Block comments start with `# ` | `# Standard retriever` |
| E501 | Max line length of 79 characters | Long strings use wrapping |

### Best Practices Used

| Practice | Why | Example |
|----------|-----|---------|
| f-strings | Clean string formatting | `f"#{i + 1}: {doc.page_content[:60]}..."` |
| `enumerate()` | Index + value in loops | `for i, doc in enumerate(results)` |
| List comprehensions | Concise list building | `[doc.metadata['id'] for doc in results]` |
| Tuple unpacking | Clean loop over pairs | `for name, ret in strategies` |
| `[:60]` slicing | Truncate long strings | `doc.page_content[:60]` |
| Docstrings | Explain function purpose | `semantic_search()` docstring |
| Descriptive names | Code reads like English | `mmr_retriever` not `mr`, `standard_results` not `sr` |

### Quick PEP 8 Cheat Sheet

```python
# GOOD (PEP 8 compliant)
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 20, "lambda_mult": 0.5},
)
for i, doc in enumerate(results):
    print(f"  #{i + 1}: {doc.page_content[:60]}...")

# BAD (violates PEP 8)
mr = vectorstore.as_retriever(search_type="mmr",search_kwargs={"k":5})
for i,d in enumerate(results):                    # no space after comma
    print(str(i+1)+": "+d.page_content[:60])      # string concat, no f-string
```